# 04 — Train LoRA on MedSAM2

Vanilla LoRA adapters on the Hiera attention QKV / output projections and the memory-attention blocks. Trains only ~0.6% of parameters.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
from cfrp_medsam2.train import TrainConfig, train
from cfrp_medsam2.model import ModelConfig
from cfrp_medsam2.lora import LoRAConfig

REPO = Path('..').resolve()

In [ ]:
cfg = TrainConfig(
    regime='lora',
    model=ModelConfig(
        backend='medsam2',
        checkpoint=str(REPO / 'checkpoints' / 'sam2.1_hiera_tiny.pt'),
        image_size=512,
    ),
    lora=LoRAConfig(
        rank=8,
        alpha=16.0,
        dropout=0.05,
        target_substrings=('qkv', 'q_proj', 'k_proj', 'v_proj', 'out_proj', 'proj'),
        exclude_substrings=('mask_decoder.iou_prediction_head', 'mlp', 'obj_ptr'),
        use_conv=False,
        train_mask_decoder=True,
        include_memory_attention=True,
    ),
    train_volumes=tuple(str(p) for p in sorted((REPO / 'data' / 'processed' / 'toy').glob('train_*.npz'))),
    val_volumes=tuple(str(p) for p in sorted((REPO / 'data' / 'processed' / 'toy').glob('val_*.npz'))),
    image_size=512,
    batch_size=1,
    epochs=5,
    lr=1.0e-4,
    prompt_mode='mixed',
    ckpt_dir=str(REPO / 'checkpoints'),
    log_path=str(REPO / 'logs' / 'lora.csv'),
)
result = train(cfg)
print('best_val_dice =', result['best_val_dice'])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv(REPO / 'logs' / 'lora.csv')
df.plot(x='epoch', y=['train_dice', 'val_dice'], figsize=(6, 3.5))
plt.title('LoRA training curves')
plt.grid(True, alpha=0.3)